In [1]:
import numpy as np
import torch
import pandas as pd
from anndata import AnnData
import scanpy as sc

In [25]:
from pathlib import Path
import warnings

import h5py
from scipy import sparse
import numpy as np
from math import ceil

import anndata as ad
from anndata.experimental import read_elem, sparse_dataset, write_elem
import os
import numba
import pyarrow as pa
import pandas as pd

import dask.array as da
import dask.dataframe as dd
from dask import delayed
import zarr
import dask
from os.path import join
from scipy.sparse import issparse
from sklearn.utils import sparsefuncs
from tqdm import tqdm
import pyarrow.parquet as pq
import pyarrow

import scanpy as sc

from dask.diagnostics import ProgressBar

warnings.filterwarnings('ignore', category=UserWarning)

In [26]:
modality_dict = {
    'dissociated': 3,
    'spatial': 4,}

specie_dict = {
    'human': 5,
    'Homo sapiens': 5,
    'Mus musculus': 6,
    'mouse': 6,}

technology_dict = {
    "merfish": 7,
    "cosmx": 8,
    "visium": 9,
    "10x 5' v2": 10,
    "10x 3' v3": 11,
    "10x 3' v2": 12,
    "10x 5' v1": 13,
    "10x 3' v1": 14,
    "10x 3' transcription profiling": 15, 
    "10x transcription profiling": 15,
    "10x 5' transcription profiling": 16,
    "CITE-seq": 17, 
    "Smart-seq v4": 18,
}

In [27]:
technology_dict

{'merfish': 7,
 'cosmx': 8,
 'visium': 9,
 "10x 5' v2": 10,
 "10x 3' v3": 11,
 "10x 3' v2": 12,
 "10x 5' v1": 13,
 "10x 3' v1": 14,
 "10x 3' transcription profiling": 15,
 '10x transcription profiling': 15,
 "10x 5' transcription profiling": 16,
 'CITE-seq': 17,
 'Smart-seq v4': 18}

## Splitting function

In [28]:
def get_split(samples, val_split: float = 0.15, test_split: float = 0.15, seed = 1):
    rng = np.random.default_rng(seed=seed)

    samples = np.array(samples)
    rng.shuffle(samples)
    n_samples = len(samples)

    n_samples_val = ceil(val_split * n_samples)
    n_samples_test = ceil(test_split * n_samples)
    n_samples_train = n_samples - n_samples_val - n_samples_test

    return {
        'train': samples[:n_samples_train],
        'val': samples[n_samples_train:(n_samples_train + n_samples_val)],
        'test': samples[(n_samples_train + n_samples_val):]
    }


## Tokenization functions

In [5]:
def sf_normalize(X):
    X = X.copy()
    counts = np.array(X.sum(axis=1))
    # avoid zero devision error
    counts += counts == 0.
    # normalize to 10000. counts
    scaling_factor = 10000. / counts

    if issparse(X):
        sparsefuncs.inplace_row_scale(X, scaling_factor)
    else:
        np.multiply(X, scaling_factor.reshape((-1, 1)), out=X)

    return X


@numba.jit(nopython=True, nogil=True)
def _sub_tokenize_data(x: np.array, max_seq_len: int = -1, aux_tokens: int = 30):
    scores_final = np.empty((x.shape[0], max_seq_len if max_seq_len > 0 else x.shape[1]))
    for i, cell in enumerate(x):
        nonzero_mask = np.nonzero(cell)[0]    
        sorted_indices = nonzero_mask[np.argsort(-cell[nonzero_mask])][:max_seq_len] 
        sorted_indices = sorted_indices + aux_tokens # we reserve some tokens for padding etc (just in case)
        if max_seq_len:
            scores = np.zeros(max_seq_len, dtype=np.int32)
        else:
            scores = np.zeros_like(cell, dtype=np.int32)
        scores[:len(sorted_indices)] = sorted_indices.astype(np.int32)
        
        scores_final[i, :] = scores
        
    return scores_final


def tokenize_data(x: np.array, median_counts_per_gene: np.array, max_seq_len: int = None):
    """Tokenize the input gene vector to a vector of 32-bit integers."""

    x = sf_normalize(x)
    median_counts_per_gene += median_counts_per_gene == 0
    out = x / median_counts_per_gene.reshape((1, -1))

    scores_final = _sub_tokenize_data(out, 4096, 30) # 20 auxiliar tokens

    return scores_final.astype('i4')


def preprocess_count_matrix(x, normalization, median_counts_per_gene):
    return x.map_blocks(
        tokenize_data, 
        median_counts_per_gene=median_counts_per_gene, 
        max_seq_len=4096, 
        dtype='f4',
        chunks=(x.chunks[0], 4096)
    )


@dask.delayed
def convert_to_dataframe(x, column_name, start, end):
    return pd.DataFrame(
        {column_name: [arr.squeeze().astype('i4') for arr in np.vsplit(x, x.shape[0])]},
        index=pd.RangeIndex(start, end)
    )

## Dask functions

In [6]:
def make_dask_chunk(x: "SparseDataset", start: int, end: int) -> da.Array:
    def take_slice(x, idx):
        return x[idx]

    return da.from_delayed(
        delayed(take_slice)(x, slice(start, end)),
        dtype=x.dtype,
        shape=(end - start, x.shape[1]),
        meta=CSRCallable,
    )
    
def sparse_dataset_as_dask(x, stride: int):
    n_chunks, rem = divmod(x.shape[0], stride)

    chunks = []
    cur_pos = 0
    for i in range(n_chunks):
        chunks.append(make_dask_chunk(x, cur_pos, cur_pos + stride))
        cur_pos += stride
    if rem:
        chunks.append(make_dask_chunk(x, cur_pos, x.shape[0]))

    return da.concatenate(chunks, axis=0)
    
def read_w_sparse_dask(group: h5py.Group | zarr.Group, obs_chunk: int = 1000) -> ad.AnnData:
    return ad.AnnData(
        X=sparse_dataset_as_dask(sparse_dataset(group["X"]), obs_chunk),
        **{
            k: read_elem(group[k]) if k in group else {}
            for k in ["layers", "obs", "var", "obsm", "varm", "uns", "obsp", "varp"]
        }
    )

def csr_callable(shape: tuple[int, int], dtype) -> sparse.csr_matrix:
    if len(shape) == 0:
        shape = (0, 0)
    if len(shape) == 1:
        shape = (shape[0], 0)
    elif len(shape) == 2:
        pass
    else:
        raise ValueError(shape)

    return sparse.csr_matrix(shape, dtype=dtype)

class CSRCallable:
    """Dummy class to bypass dask checks"""
    def __new__(cls, shape, dtype):
        return csr_callable(shape, dtype)

def index_delayed_sparse(x: "SparseDataset", idx) -> da.Array:
    def take_slice(x, idx):
        return x[idx, :]

    return da.from_delayed(
        delayed(take_slice)(x, idx),
        dtype=x.dtype,
        shape=(len(idx), x.shape[1]),
        meta=CSRCallable,
    )

def to_64bit_indptr(x: sparse.spmatrix):
    x.indptr = x.indptr.astype(np.int64)
    return x

## Code

The flow is the following:
- Concatenate anndatas
- Do outer join if needed (probably better done that beforehand)
- Compute separate means for technology
- Tokenize separately for technology
- Concatenate again anndatas
- Shuffle anndata and split in train and test

In [7]:
DATA_DIR = Path("/lustre/groups/ml01/projects/2023_nicheformer/data/cellxgene_census_raw")
#DATA_DIR = Path("/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_corpus/cellxgene-data-public/cell-census/2023-12-06/h5ads/")

files = [f for f in os.listdir(DATA_DIR) if os.path.isfile(os.path.join(DATA_DIR, f))]

PTHS = [DATA_DIR / p for p in files]


KeyboardInterrupt



In [7]:
OUT_PATH = "/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_to_tokenize"
split = "train"

In [3]:
model = sc.read_h5ad('/lustre/groups/ml01/projects/2023_nicheformer/data/data_to_tokenize/model.h5ad')

In [29]:
model

AnnData object with n_obs × n_vars = 1 × 20310
    obs: 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id', 'assay', 'cell_type', 'development_stage', 'disease', 'tissue', 'tissue_general', 'specie', 'technology', 'dataset', 'x', 'y', 'assay_ontology_term_id', 'sex_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'condition_id', 'tissue_type', 'library_key', 'organism', 'sex', 'niche', 'region', 'nicheformer_split', 'author_cell_type', 'batch'

SUPER ANNOYING. 
We need to prepare the dataset to be tokenized one by one, this means read a dataset, expand their gene set using an outer join and store it again. Also, it runs OOM so it needs to be done sequentially.

In [ ]:
success = ["/lustre/groups/ml01/projects/2023_nicheformer/data/cellxgene_census_raw/4.h5ad", "/lustre/groups/ml01/projects/2023_nicheformer/data/cellxgene_census_raw/mouse_16.h5ad"]

for p in tqdm(PTHS[50:]):
    print(f"Reading {p}")
    adata = sc.read_h5ad(p)
    print(adata)
    adata.var.index = adata.var.feature_id
    adata = ad.concat([model, adata], join='outer', axis=0)
    adata = adata[1:]
    adata.obs['is_primary_data'] = adata.obs['is_primary_data'].astype(bool)
    adata.write_h5ad(os.path.join(OUT_PATH, p.name))
    print(adata)
    print(f"Finished {p}")
    success.append(p)
    print(f"Success: {p}")  

Change paths again.

In [7]:
DATA_DIR = Path("/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_to_tokenize")
#DATA_DIR = Path("/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_corpus/cellxgene-data-public/cell-census/2023-12-06/h5ads/")

files = [f for f in os.listdir(DATA_DIR) if os.path.isfile(os.path.join(DATA_DIR, f))]

PTHS = [DATA_DIR / p for p in files]

In [8]:
OUT_PATH = "/lustre/groups/ml01/projects/2023_nicheformer/data/nicheformer_tokens"
split = "train"

In [9]:
dask_adatas = []

for p in tqdm(PTHS):
    try:
        dask_adatas.append(read_w_sparse_dask(h5py.File(p), 10_000))
    except Exception as e:
        print(f"Error reading {p}: {e}")
        pass


 45%|████▍     | 115/257 [01:49<01:39,  1.43it/s]

Error reading /lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_to_tokenize/dissociated_mean_script.npy: Unable to open file (file signature not found)


100%|██████████| 257/257 [04:30<00:00,  1.05s/it]


Standardize name of genes.

Read already standardised file.

In [10]:
combined = ad.concat(dask_adatas, axis=0)

In [11]:
combined

AnnData object with n_obs × n_vars = 57055472 × 20310
    obs: 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id', 'assay', 'cell_type', 'development_stage', 'disease', 'tissue', 'tissue_general', 'specie', 'technology'

In [70]:
testing = read_w_sparse_dask(h5py.File('/lustre/groups/ml01/projects/2023_nicheformer_data_anna.schaar/concat/single_files/human_mouse_cosmx_v2.h5ad'), 10_000)

In [71]:
testing

AnnData object with n_obs × n_vars = 2115437 × 1857
    obs: 'dataset', 'x', 'y', 'assay_ontology_term_id', 'sex_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'donor_id', 'condition_id', 'tissue_type', 'library_key', 'assay', 'organism', 'sex', 'tissue', 'niche', 'region', 'nicheformer_split', 'batch'
    var: 'level_0-1', 'index-1', 'DisplayName-1', 'feature_is_filtered-1', 'feature_name-1', 'feature_reference-1', 'feature_biotype-1', 'Gene name-1', 'Source of gene name-1', 'Human gene stable ID-1', 'Human gene name-1', 'Human orthology confidence [0 low, 1 high]-1'
    uns: 'nicheformer_version', 'schema_version'

In [30]:
merfish = sc.read_h5ad('/lustre/groups/ml01/projects/2023_nicheformer_data_anna.schaar/concat/single_files/human_mouse_cosmx_v2.h5ad')
merfish

AnnData object with n_obs × n_vars = 2115437 × 1857
    obs: 'dataset', 'x', 'y', 'assay_ontology_term_id', 'sex_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'donor_id', 'condition_id', 'tissue_type', 'library_key', 'assay', 'organism', 'sex', 'tissue', 'niche', 'region', 'nicheformer_split', 'batch'
    var: 'level_0-1', 'index-1', 'DisplayName-1', 'feature_is_filtered-1', 'feature_name-1', 'feature_reference-1', 'feature_biotype-1', 'Gene name-1', 'Source of gene name-1', 'Human gene stable ID-1', 'Human gene name-1', 'Human orthology confidence [0 low, 1 high]-1'
    uns: 'nicheformer_version', 'schema_version'

In [31]:
adata = ad.concat([model, merfish], join='outer', axis=0)

In [32]:
adata

AnnData object with n_obs × n_vars = 2115438 × 20310
    obs: 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id', 'assay', 'cell_type', 'development_stage', 'disease', 'tissue', 'tissue_general', 'specie', 'technology', 'dataset', 'x', 'y', 'assay_ontology_term_id', 'sex_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'condition_id', 'tissue_type', 'library_key', 'organism', 'sex', 'niche', 'region', 'nicheformer_split', 'author_cell_type', 'batch'

In [14]:
set(list(adata.var_names)).difference(set(list(model.var_names)))

set()

In [13]:
adapted = list(adata.var_names)
adapted.remove('ENSG00000229415')

ValueError: list.remove(x): x not in list

In [15]:
len(adapted)

NameError: name 'adapted' is not defined

In [17]:
adata = adata[:, adapted]

NameError: name 'adapted' is not defined

In [33]:
merfish = adata[1:]

In [34]:
merfish.obs = merfish.obs[['assay', 'organism', 'nicheformer_split', 'batch']]

In [35]:
merfish

AnnData object with n_obs × n_vars = 2115437 × 20310
    obs: 'assay', 'organism', 'nicheformer_split', 'batch'

In [37]:
merfish.write('/lustre/groups/ml01/projects/2023_nicheformer/data/data_to_tokenize/cosmx_ready_to_tokenize.h5ad')

In [62]:
(combined.var_names == merfish.var_names).all()

True

In [16]:
combined = ad.concat(dask_adatas, axis=0)

In [17]:
combined

AnnData object with n_obs × n_vars = 57055472 × 20310
    obs: 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id', 'assay', 'cell_type', 'development_stage', 'disease', 'tissue', 'tissue_general', 'specie', 'technology'

In [13]:
combined.obs.soma_joinid.unique()

array([      nan,  6042010.,  6042011., ..., 50775131., 50775132.,
       50775133.])

In [14]:
combined.obs = combined.obs.rename(columns={'technology': 'modality'})

In [15]:
combined.obs['modality'] = 'dissociated'

In [16]:
print(combined.obs.specie.unique())
print(combined.obs.modality.unique())
print(combined.obs.assay.unique())

['Mus musculus', 'Homo sapiens', 'human', 'mouse']
Categories (4, object): ['Homo sapiens', 'Mus musculus', 'human', 'mouse']
['dissociated']
['10x 3' v3', '10x 5' transcription profiling', '10x 3' v2', NaN, '10x 5' v1', ..., '10x 5' v2', '10x 3' v1', 'Smart-seq v4', '10x transcription profiling', 'CITE-seq']
Length: 11
Categories (10, object): ['10x 3' transcription profiling', '10x 3' v1', '10x 3' v2', '10x 3' v3', ..., '10x 5' v2', '10x transcription profiling', 'CITE-seq', 'Smart-seq v4']


In [21]:
print(combined.obs.specie.unique())
print(combined.obs.modality.unique())
print(combined.obs.assay.unique())

['Mus musculus', 'Homo sapiens', 'human', 'mouse']
Categories (4, object): ['Homo sapiens', 'Mus musculus', 'human', 'mouse']
['dissociated']
['10x 3' v3', '10x 5' transcription profiling', '10x 3' v2', NaN, '10x 5' v1', ..., '10x 5' v2', '10x 3' v1', 'Smart-seq v4', '10x transcription profiling', 'CITE-seq']
Length: 11
Categories (10, object): ['10x 3' transcription profiling', '10x 3' v1', '10x 3' v2', '10x 3' v3', ..., '10x 5' v2', '10x transcription profiling', 'CITE-seq', 'Smart-seq v4']


In [17]:
combined.obs['assay'] = combined.obs['assay'].fillna("10x 3' v3")

In [18]:
combined.obs

,soma_joinid,is_primary_data,dataset_id,donor_id,assay,cell_type,development_stage,disease,tissue,tissue_general,specie,modality
AAACCCAAGCAAGTGC-1_0,NaN,NaN,NaN,GSM4563822_d7immob,10x 3' v3,NaN,NaN,NaN,tendon,NaN,Mus musculus,dissociated
AAACCCACAAGAAACT-1_0,NaN,NaN,NaN,GSM4563822_d7immob,10x 3' v3,NaN,NaN,NaN,tendon,NaN,Mus musculus,dissociated
AAACCCAGTGTCCATA-1_0,NaN,NaN,NaN,GSM4563822_d7immob,10x 3' v3,NaN,NaN,NaN,tendon,NaN,Mus musculus,dissociated
AAACCCAGTTCTCCCA-1_0,NaN,NaN,NaN,GSM4563822_d7immob,10x 3' v3,NaN,NaN,NaN,tendon,NaN,Mus musculus,dissociated
AAACCCATCAGCACCG-1_0,NaN,NaN,NaN,GSM4563822_d7immob,10x 3' v3,NaN,NaN,NaN,tendon,NaN,Mus musculus,dissociated
...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCAAGGCAGTCA-1_2,NaN,NaN,NaN,GSM4175979,10x transcription profiling,NaN,NaN,NaN,pancreas,NaN,Mus musculus,dissociated
TTTGTCAAGGCTCAGA-1_2,NaN,NaN,NaN,GSM4175979,10x transcription profiling,NaN,NaN,NaN,pancreas,NaN,Mus musculus,dissociated
TTTGTCAAGGGTGTTG-1_2,NaN,NaN,NaN,GSM4175979,10x transcription profiling,NaN,NaN,NaN,pancreas,NaN,Mus musculus,dissociated
TTTGTCATCATTTGGG-1_2,NaN,NaN,NaN,GSM4175979,10x transcription profiling,NaN,NaN,NaN,pancreas,NaN,Mus musculus,dissociated


In [19]:
obs = combined.obs

In [20]:
obs.reset_index(inplace=True)

In [21]:
len(obs['index'].unique())

19200138

In [22]:
obs.specie.unique()

['Mus musculus', 'Homo sapiens', 'human', 'mouse']
Categories (4, object): ['Homo sapiens', 'Mus musculus', 'human', 'mouse']

Features to integers.

In [23]:
obs_train = obs.copy()

In [24]:
obs_train.columns

Index(['index', 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id',
       'assay', 'cell_type', 'development_stage', 'disease', 'tissue',
       'tissue_general', 'specie', 'modality'],
      dtype='object')

In [25]:
obs_train.replace({'specie': specie_dict}, inplace=True)
obs_train.replace({'assay': technology_dict}, inplace=True)
obs_train.replace({'modality': modality_dict}, inplace=True)

In [26]:
obs_train

,index,soma_joinid,is_primary_data,dataset_id,donor_id,assay,cell_type,development_stage,disease,tissue,tissue_general,specie,modality
0,AAACCCAAGCAAGTGC-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
1,AAACCCACAAGAAACT-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
2,AAACCCAGTGTCCATA-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
3,AAACCCAGTTCTCCCA-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
4,AAACCCATCAGCACCG-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57055467,TTTGTCAAGGCAGTCA-1_2,NaN,NaN,NaN,GSM4175979,15,NaN,NaN,NaN,pancreas,NaN,6,3
57055468,TTTGTCAAGGCTCAGA-1_2,NaN,NaN,NaN,GSM4175979,15,NaN,NaN,NaN,pancreas,NaN,6,3
57055469,TTTGTCAAGGGTGTTG-1_2,NaN,NaN,NaN,GSM4175979,15,NaN,NaN,NaN,pancreas,NaN,6,3
57055470,TTTGTCATCATTTGGG-1_2,NaN,NaN,NaN,GSM4175979,15,NaN,NaN,NaN,pancreas,NaN,6,3


In [27]:
print(obs_train.specie.unique())
print(obs_train.modality.unique())
print(obs_train.assay.unique())

[6, 5]
Categories (2, int64): [6, 5]
[3]
[11, 16, 12, 13, 15, 10, 14, 18, 17]
Categories (9, int64): [15, 14, 12, 11, ..., 13, 10, 17, 18]


In [28]:
lookup_path = join(OUT_PATH, 'categorical_lookup')
if not os.path.exists(lookup_path):
    os.makedirs(lookup_path)

In [29]:
for col in obs_train.columns:
    if obs_train[col].dtype.name == 'category':
        cats_train = pd.Series(dict(enumerate(obs_train[col].cat.categories))).to_frame().rename(columns={0: 'label'})
cats_train.to_parquet(join(lookup_path, f'{col}.parquet'), index=True)

In [30]:
obs_train

,index,soma_joinid,is_primary_data,dataset_id,donor_id,assay,cell_type,development_stage,disease,tissue,tissue_general,specie,modality
0,AAACCCAAGCAAGTGC-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
1,AAACCCACAAGAAACT-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
2,AAACCCAGTGTCCATA-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
3,AAACCCAGTTCTCCCA-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
4,AAACCCATCAGCACCG-1_0,NaN,NaN,NaN,GSM4563822_d7immob,11,NaN,NaN,NaN,tendon,NaN,6,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57055467,TTTGTCAAGGCAGTCA-1_2,NaN,NaN,NaN,GSM4175979,15,NaN,NaN,NaN,pancreas,NaN,6,3
57055468,TTTGTCAAGGCTCAGA-1_2,NaN,NaN,NaN,GSM4175979,15,NaN,NaN,NaN,pancreas,NaN,6,3
57055469,TTTGTCAAGGGTGTTG-1_2,NaN,NaN,NaN,GSM4175979,15,NaN,NaN,NaN,pancreas,NaN,6,3
57055470,TTTGTCATCATTTGGG-1_2,NaN,NaN,NaN,GSM4175979,15,NaN,NaN,NaN,pancreas,NaN,6,3


In [31]:
obs_train.dtypes

index                  object
soma_joinid           float64
is_primary_data        object
dataset_id           category
donor_id             category
assay                category
cell_type            category
development_stage    category
disease              category
tissue               category
tissue_general       category
specie               category
modality                int64
dtype: object

In [32]:
obs_train['specie'] = obs_train['specie'].astype(int)
obs_train['assay'] = obs_train['assay'].astype(int)

In [33]:
# only use integer labels from now on
for col in obs_train.columns:
    if obs_train[col].dtype.name == 'category':
        obs_train[col] = obs_train[col].cat.codes.astype('i8')

In [34]:
obs_train

,index,soma_joinid,is_primary_data,dataset_id,donor_id,assay,cell_type,development_stage,disease,tissue,tissue_general,specie,modality
0,AAACCCAAGCAAGTGC-1_0,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
1,AAACCCACAAGAAACT-1_0,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
2,AAACCCAGTGTCCATA-1_0,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
3,AAACCCAGTTCTCCCA-1_0,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
4,AAACCCATCAGCACCG-1_0,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57055467,TTTGTCAAGGCAGTCA-1_2,NaN,NaN,-1,2262,15,-1,-1,-1,174,-1,6,3
57055468,TTTGTCAAGGCTCAGA-1_2,NaN,NaN,-1,2262,15,-1,-1,-1,174,-1,6,3
57055469,TTTGTCAAGGGTGTTG-1_2,NaN,NaN,-1,2262,15,-1,-1,-1,174,-1,6,3
57055470,TTTGTCATCATTTGGG-1_2,NaN,NaN,-1,2262,15,-1,-1,-1,174,-1,6,3


In [35]:
obs_train.columns

Index(['index', 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id',
       'assay', 'cell_type', 'development_stage', 'disease', 'tissue',
       'tissue_general', 'specie', 'modality'],
      dtype='object')

In [36]:
obs_train = obs_train[['soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id',
       'assay', 'cell_type', 'development_stage', 'disease', 'tissue',
       'tissue_general', 'specie', 'modality']]

In [37]:
obs_train

,soma_joinid,is_primary_data,dataset_id,donor_id,assay,cell_type,development_stage,disease,tissue,tissue_general,specie,modality
0,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
1,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
2,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
3,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
4,NaN,NaN,-1,2392,11,-1,-1,-1,246,-1,6,3
...,...,...,...,...,...,...,...,...,...,...,...,...
57055467,NaN,NaN,-1,2262,15,-1,-1,-1,174,-1,6,3
57055468,NaN,NaN,-1,2262,15,-1,-1,-1,174,-1,6,3
57055469,NaN,NaN,-1,2262,15,-1,-1,-1,174,-1,6,3
57055470,NaN,NaN,-1,2262,15,-1,-1,-1,174,-1,6,3


### If your data is big and you run OOM
(maybe skip this part)

Write memory-mapped file. This file allows reading random (shuffled) indices faster with OOM errors.

We write a dissociated memory-mapped file and a spatial memory-mapped file cause they require different tokenizations.

In [24]:
with h5py.File("/lustre/groups/ml01/projects/2023_nicheformer/data/memory_mapped_files/dissociated.h5", "w") as f:
    ad.experimental.write_elem(f, "X", combined[dissociated_indices].X.map_blocks(to_64bit_indptr, dtype='float32'))

with h5py.File("/lustre/groups/ml01/projects/2023_nicheformer/data/memory_mapped_files/spatial.h5", "w") as f:
    ad.experimental.write_elem(f, "X", combined[spatial_indices].X.map_blocks(to_64bit_indptr, dtype='float32'))

/home/icb/alejandro.tejada/miniconda3/envs/tokenizer/lib/python3.10/site-packages/anndata/_core/index.py:158: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return a[subset_idx]


This is just to cast the indprt and makes the loading faster afterwards. Some hacky code, maybe better change at some point (not now).

In [24]:
f = h5py.File("/lustre/groups/ml01/projects/2023_nicheformer/data/memory_mapped_files/dissociated.h5")
X_h5_dissociated = ad.experimental.sparse_dataset(f["X"])._to_backed()
%time X_h5_dissociated[0]
%time X_h5_dissociated[0]

CPU times: user 239 µs, sys: 1.46 ms, total: 1.69 ms
Wall time: 128 ms
CPU times: user 172 µs, sys: 494 µs, total: 666 µs
Wall time: 955 µs


<1x19331 sparse matrix of type '<class 'numpy.float32'>'
	with 3099 stored elements in Compressed Sparse Row format>

In [ ]:
f = h5py.File("/lustre/groups/ml01/projects/2023_nicheformer/data/memory_mapped_files/spatial.h5")
X_h5_spatial = ad.experimental.sparse_dataset(f["X"])._to_backed()
%time X_h5_spatial[0]
%time X_h5_spatial[0]

Now X is just a np.array, so we use the direct tokenizer function.

First let's compute the non-zero medians.

DON'T RUN THIS.

In [ ]:
spatial_data = X_h5_spatial.A

# Convert zero values to NaN
spatial_data[spatial_data == 0] = np.nan

# Compute the median along each column, ignoring NaN values
spatial_medians = np.nanmedian(spatial_data, axis=1)

In [ ]:
dissociated_data = X_h5_dissociated.A

# Convert zero values to NaN
dissociated_data[dissociated_data == 0] = np.nan

# Compute the median along each column, ignoring NaN values
dissociated_medians = np.nanmedian(dissociated_data, axis=1)

We want the data to be shuffled, so we need to retrieve random indices (that's the reason why the memory-mapped file is needed). As we have 2 of those files, we need to retrieve random indices from them independently.

In [26]:
obs_dissociated = obs.loc[dissociated_indices, :]
obs_spatial = obs.loc[spatial_indices, :]

We need to reset the indices to properly retrieve the indices.

In [27]:
obs_dissociated.reset_index(drop=True, inplace=True)
obs_spatial.reset_index(drop=True, inplace=True)

Randomize indices.

In [28]:
dissociated_random_indices = obs.loc[dissociated_indices, :].index.to_numpy()
rng = np.random.default_rng(seed=1)
dissociated_random_indices = np.array(rng.permutation(dissociated_random_indices))


# Spatial probably needs to some other splitting for train and test
spatial_random_indices = obs.loc[spatial_indices, :].index.to_numpy()
rng = np.random.default_rng(seed=1)
spatial_random_indices = np.array(rng.permutation(spatial_random_indices))

In [39]:
## Write store
CHUNK_SIZE = 57877
ROW_GROUP_SIZE = 1024
N_BATCHES = combined.n_obs // CHUNK_SIZE

Random medians just to test.

In [31]:
dissociated_medians = np.random.randint(0, 20, (1, 19331))
spatial_medians = np.random.randint(0, 20, (1, 19331))

In [37]:
batch_dissociated_indices = np.array_split(dissociated_random_indices, N_BATCHES)
batch_spatial_indices = np.array_split(spatial_random_indices, N_BATCHES)

In [ ]:
for i in tqdm(range(N_BATCHES)):
    
    random_indices_dissociated = batch_dissociated_indices[i]
    random_indices_spatial = batch_spatial_indices[i]
    
    X_shuffled_dissociated = X_h5_dissociated[random_indices_dissociated]
    X_shuffled_dissociated = tokenize_data(X_shuffled_dissociated, dissociated_medians, 4096)

    X_shuffled_spatial = X_h5_spatial[random_indices_spatial]
    X_shuffled_spatial = tokenize_data(X_shuffled_spatial, spatial_medians, 4096)

    obs_dissociated = obs_dissociated.loc[random_indices_dissociated, :]
    obs_spatial = obs_spatial.loc[random_indices_spatial, :]

    X_shuffled = np.concantenate((X_shuffled_dissociated, X_shuffled_spatial), axis=1)
    obs_shuffled = pd.concat([obs_dissociated, obs_spatial], axis=1)

    # create pandas dataframe from np.array
    X_shuffled_df = pd.DataFrame({'X': [X_shuffled[i, :] for i in range(X_shuffled.shape[0])]})

    # concatenate dataframes
    total = pd.concat([X_shuffled_df, obs_dissociated], axis=1)

    # mix spatial and dissociate data
    total = total.sample(frac=1)

    print("tokenized")
    
    total_table = pyarrow.Table.from_pandas(total)

    pq.write_table(total_table, f'{join(OUT_PATH, split)}/train-{i}.parquet',
                    row_group_size=1024,)

## If that takes too long (can be)

Shortcut: generate all data without randomizing it and later mix the parquet files

In [1]:
import numpy as np

In [2]:
mean_dissociated = np.load("/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_to_tokenize/dissociated_mean_script.npy")
mean_dissociated


array([ 2.57534268,  1.62529228,  2.06693628, ...,  6.26075084,
       12.06460517,  2.65934034])

In [42]:
combined

AnnData object with n_obs × n_vars = 57055472 × 20310
    obs: 'index', 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id', 'assay', 'cell_type', 'development_stage', 'disease', 'tissue', 'tissue_general', 'specie', 'modality'

In [ ]:
mean_dissociated = combined.X.map_blocks(lambda x: sf_normalize(x.toarray()), dtype=np.float32)
zero_to_nan = np.where(mean_dissociated == 0.0, np.nan, mean_dissociated)
mean_dissociated = np.nanmedian(zero_to_nan, axis=1)

mean_dissociated = mean_dissociated.compute()

In [ ]:
mean_dissociated.shape

In [41]:
mean_dissociated = combined.X.map_blocks(lambda x: sf_normalize(x.toarray()), dtype=np.float32).mean(axis=0)
mean_dissociated = mean_dissociated.compute()


KeyboardInterrupt



In [58]:
mean_dissociated = np.nan_to_num(mean_dissociated)
rounded_values = np.where((mean_dissociated % 1) >= 0.5, np.ceil(mean_dissociated), np.floor(mean_dissociated))
mean_dissociated = np.where(rounded_values == 0, 1, rounded_values)

In [59]:
nan_indices = np.where(np.isnan(mean_dissociated))[0]

# Count the number of NaN values
nan_count = len(nan_indices)

if nan_count > 0:
    print(f"The vector contains {nan_count} NaN value(s) at indices: {nan_indices}")
else:
    print("The vector does not contain any NaN values.")

The vector does not contain any NaN values.


In [113]:
np.save("/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_to_tokenize/dissociated_mean.npy", mean_dissociated)


In [66]:
## Write store
CHUNK_SIZE = 57877
ROW_GROUP_SIZE = 1024
N_BATCHES = combined.n_obs // CHUNK_SIZE

In [67]:
N_BATCHES

985

In [71]:
X = preprocess_count_matrix(combined.X.map_blocks(lambda x: x.toarray(), dtype=np.float32), "dissociated", mean_dissociated)

# add an index column to identifiy each sample
obs_train['idx'] = np.arange(len(obs_train), dtype='i8')
start_index = [0] + list(np.cumsum(X.chunks[0]))[:-1]
end_index = list(np.cumsum(X.chunks[0]))

# calculate divisons for dask dataframe
divisions = [0] + list(np.cumsum(X.chunks[0]))
divisions[-1] = divisions[-1] - 1

print(f'{X.shape[0]} cells')

ddf = dd.from_delayed(
    [
        convert_to_dataframe(arr, 'X', start, end) for arr, start, end in 
        zip(X.to_delayed().flatten().tolist(), start_index, end_index)
    ],
    divisions=divisions,
    meta={'X': 'object'}
)

obs_train = obs_train[['assay', 'specie', 'modality', 'idx']] 

obs_dask = dd.from_pandas(obs_train, chunksize=CHUNK_SIZE)

ddf = dd.multi.concat([ddf, obs_dask], axis=1)

schema = pa.schema([
    ('X', pa.list_(pa.int32())),
    ('assay', pa.int64()),
    ('idx', pa.int64()),
    ('specie', pa.int64()),
    ('modality', pa.int64()),
])


ddf.to_parquet(
    join(OUT_PATH, split), 
    engine='pyarrow',
    schema=schema,
    write_metadata_file=True,
    name_function=lambda j: f"tokens-{j}.parquet",
    row_group_size=ROW_GROUP_SIZE
)

57055472 cells


In [123]:
PTHS[0]

PosixPath('/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_to_tokenize/GSE150995.h5ad')

In [121]:
aa = pd.read_parquet('/lustre/groups/ml01/projects/2023_nicheformer/data/final_tokens/train/tokens-0.parquet')

In [122]:
aa

,X,assay,idx,specie,modality
0,"[3568, 12181, 4571, 11244, 4111, 17417, 11241,...",11,0,6,3
1,"[20280, 16990, 269, 7238, 8103, 343, 16266, 61...",11,1,6,3
2,"[11244, 3568, 381, 309, 4136, 5199, 12181, 771...",11,2,6,3
3,"[20211, 20235, 17560, 18286, 17571, 20240, 307...",11,3,6,3
4,"[6224, 6223, 12178, 2532, 8338, 14350, 13355, ...",11,4,6,3
...,...,...,...,...,...
9995,"[17417, 8939, 309, 4053, 8642, 4663, 5640, 724...",11,9995,6,3
9996,"[13382, 12483, 13097, 7357, 12569, 12783, 1855...",11,9996,6,3
9997,"[3568, 11244, 12181, 14701, 4116, 6629, 4395, ...",11,9997,6,3
9998,"[668, 3568, 11244, 12181, 20280, 4395, 6225, 1...",11,9998,6,3


In [2]:
OUT_PATH = "/lustre/groups/ml01/projects/2023_nicheformer/data/final_tokens/train"

In [ ]:
all_files = os.listdir(join(OUT_PATH))

for i, file in enumerate(all_files):
    if file.endswith('parquet'):
        first_data = pd.read_parquet(join(OUT_PATH,file))
        print(i, first_data.shape[0])
        

In [ ]:
# List all files in the folder
n = 5

all_files = os.listdir(join(OUT_PATH, split))

# Do mixing n times
for subset in tqdm(range(5)):
    for i in tqdm(range(all_files)):

        # Generate two random numbers from a uniform distribution between 0 and 1
        random_numbers = random_numbers = np.random.uniform(0, all_files, 2)
        
        random_number1 = random_numbers[0]
        random_number2 = random_numbers[1]
        
        # Read the parquet files
        first_data = pd.read_parquet(f'/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_tokenized/dissociated-{i}.parquet')
        second_data = pd.read_parquet(f'/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_tokenized/dissociated-{random_number1}.parquet')
        thid_data = pd.read_parquet(f'/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_tokenized/dissociated-{random_number2}.parquet')

        # Merge the data
        merged_data = pd.concat([first_data, second_data, thid_data])

        # Shuffle the merged data to mix the information
        merged_data = merged_data.sample(frac=1).reset_index(drop=True)

        # Split the merged data into 3 parts
        split_point = len(merged_data) // 3
        mixed_first_data = merged_data.iloc[:split_point]
        mixed_second_data = merged_data.iloc[split_point:2*split_point]
        mixed_third_data = merged_data.iloc[-split_point:]
        
        # Write the mixed data back to parquet files
        mixed_first_table = pyarrow.Table.from_pandas(mixed_first_data)
        mixed_second_table = pyarrow.Table.from_pandas(mixed_second_data)
        mixed_third_table = pyarrow.Table.from_pandas(mixed_third_data)

        pq.write_table(mixed_first_table, f'/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_tokenized/dissociated-{i}.parquet',
                    row_group_size=1024,)
        pq.write_table(mixed_second_table, f'/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_tokenized/dissociated-{random_number1}.parquet',
                    row_group_size=1024,)
        pq.write_table(mixed_thid_table, f'/lustre/groups/ml01/projects/2023_nicheformer/data/dissociated_tokenized/dissociated-{random_number2}.parquet',
                    row_group_size=1024,)